# 05 - Model Building

Trains and compares 8 classification algorithms on the engineered feature set, using stratified 5-fold cross-validation and a held-out test set.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from train_model import run_training

results = run_training()
print('Best model:', results['best_name'])

2026-07-07 08:13:04,058 | INFO | Logistic Regression: Test AUC=0.7835 | CV AUC=0.7880 +/- 0.0118


2026-07-07 08:13:04,268 | INFO | Decision Tree: Test AUC=0.7522 | CV AUC=0.7494 +/- 0.0104


2026-07-07 08:13:16,365 | INFO | Random Forest: Test AUC=0.7707 | CV AUC=0.7771 +/- 0.0115


2026-07-07 08:13:24,394 | INFO | Gradient Boosting: Test AUC=0.7801 | CV AUC=0.7840 +/- 0.0105


2026-07-07 08:13:26,752 | INFO | XGBoost: Test AUC=0.7698 | CV AUC=0.7661 +/- 0.0093


...

2026-07-07 08:14:02,246 | INFO | Saved object to /home/claude/churn/models/trained_model.pkl


2026-07-07 08:14:02,248 | INFO | Saved object to /home/claude/churn/models/feature_scaler.pkl


2026-07-07 08:14:02,251 | INFO | Saved object to /home/claude/churn/models/label_encoders.pkl


2026-07-07 08:14:02,253 | INFO | Saved object to /home/claude/churn/models/model_metadata.pkl


Best model: Logistic Regression


## Model comparison table (Test AUC + Cross-Validation AUC)

In [2]:
rows = []
for name, r in results['results'].items():
    rows.append({'Model': name, 'Test_AUC': round(r['test_auc'],4), 'CV_AUC_Mean': round(r['cv_auc_mean'],4), 'CV_AUC_Std': round(r['cv_auc_std'],4)})
comparison = pd.DataFrame(rows).sort_values('Test_AUC', ascending=False).reset_index(drop=True)
comparison

,Model,Test_AUC,CV_AUC_Mean,CV_AUC_Std
0,Logistic Regression,0.7835,0.7880,0.0118
1,Gradient Boosting,0.7801,0.7840,0.0105
2,Random Forest,0.7707,0.7771,0.0115
3,XGBoost,0.7698,0.7661,0.0093
4,Naive Bayes,0.7641,0.7700,0.0087
5,SVM,0.7550,0.7480,0.0057
6,Decision Tree,0.7522,0.7494,0.0104
7,K-Nearest Neighbors,0.7384,0.7404,0.0066


**Why these 8 models:**
- **Logistic Regression** — fast, interpretable baseline; coefficients map directly to business drivers.
- **Decision Tree** — simple, explainable rules for a non-technical audience.
- **Random Forest / Gradient Boosting / XGBoost** — ensemble methods that typically win on tabular churn data by capturing non-linear interactions (e.g. senior + no tech support).
- **SVM** — tests whether a wide-margin classifier helps on this feature space.
- **K-Nearest Neighbors** — simple distance-based baseline sensitive to feature scaling.
- **Naive Bayes** — extremely fast baseline; useful sanity check on feature independence assumptions.

**Result:** Logistic Regression narrowly wins on Test AUC, with Gradient Boosting close behind — telling us the churn signal here is largely additive/linear rather than deeply non-linear, which is itself a useful, reportable finding (and means the winning model is also the most explainable one).

## Persisted artifacts

The winning model, scaler, label encoders, and metadata have already been saved to `/models` by `train_model.py`, ready for `predict.py` or a future API layer.

In [3]:
import config
from pathlib import Path
for p in [config.MODEL_PATH, config.SCALER_PATH, config.MODELS_DIR/'label_encoders.pkl', config.MODELS_DIR/'model_metadata.pkl']:
    print(p.name, '->', Path(p).exists())

trained_model.pkl -> True
feature_scaler.pkl -> True
label_encoders.pkl -> True
model_metadata.pkl -> True
